In [6]:
// Jupyterの実行プロセスに、Gitの隠れLinuxツール（xxdやopenssl）のパスを通す
let old_path = std::env::var("PATH").unwrap_or_default();
let git_bin = r"C:\Program Files\Git\usr\bin";
// unsafe ブロックで囲むことで実行可能になります
unsafe {
    std::env::set_var("PATH", format!("{};{}", git_bin, old_path));
}

println!("[+] Windowsの環境パスにGitのツール群を追加しました。");

[+] Windowsの環境パスにGitのツール群を追加しました。


In [13]:
use std::process::Command;
use std::fs::File;
use std::io::Write;

fn main() {
    let git_openssl = r"C:\Program Files\Git\usr\bin\openssl.exe";
    println!("改行コードを考慮した、正確なJPEGヘッダの探索を開始します...");

    // 8192 から 65536 までをループ
    for n in 8192..(4096 * 16) {
        let byte4 = (n / 256) as u8; // 5バイト目
        let byte5 = (n % 256) as u8; // 6バイト目

        // 10バイトのベースデータを構築
        let mut password_bytes: Vec<u8> = vec![
            0xFF, 0xD8, 0xFF, 0xE1, 
            byte4, byte5, 
            0x45, 0x78, 0x69, 0x66 // "Exif"
        ];
        
        // ★最重要: OpenSSLの仕様（1行読み込み）に合わせるため、
        // 末尾に「改行コード (0x0A = '\n')」を明示的に付与する
        password_bytes.push(0x0A);

        // カレントディレクトリの pass.bin に上書き保存
        let mut file = File::create("pass.bin").expect("ファイル作成失敗");
        file.write_all(&password_bytes).expect("書き込み失敗");

        // OpenSSLを使って復号テスト
        let output = Command::new(git_openssl)
            .arg("enc")
            .arg("-d")
            .arg("-aes-256-cbc")
            .arg("-iter")
            .arg("10000")
            .arg("-in")
            .arg("flag.jpg.enc")
            .arg("-out")
            .arg("decrypted_flag.jpg")
            .arg("-pass")
            .arg("file:pass.bin")
            .output()
            .expect("opensslの実行に失敗しました");

        // 復号自体が成功（status.success）し、かつ「出力ファイルが本物のJPEGヘッダ」になっているかダブルチェック
        if output.status.success() {
            // 生成されたファイルの先頭2バイトをチェック
            if let Ok(mut f) = File::open("decrypted_flag.jpg") {
                let mut header = [0u8; 2];
                use std::io::Read;
                if f.read_exact(&mut header).is_ok() {
                    // JPEGの正しいマジックバイト (FF D8) であれば完全な正解！
                    if header[0] == 0xFF && header[1] == 0xD8 {
                        println!("\n[🎉] 本物のJPEG復号に成功しました！正解の値: {}", n);
                        println!("[👍] 'decrypted_flag.jpg' が綺麗に生成されました。");
                        return;
                    }
                }
            }
        }

        if n % 1000 == 0 {
            print!(".");
            let _ = std::io::stdout().flush();
        }
    }

    println!("\n[❌] 正解が見つかりませんでした。");
}

main();

改行コードを考慮した、正確なJPEGヘッダの探索を開始します...
......
[🎉] 本物のJPEG復号に成功しました！正解の値: 14672
[👍] 'decrypted_flag.jpg' が綺麗に生成されました。
